# WISE 0855 — SPHEREx DR1 inventory
Run with the project's astronomy extra. The default reads the saved IRSA response so the inventory can be reproduced offline. Set `refresh = True` to query IRSA again. The same detector can sample different wavelengths at the target across exposures; this inventory identifies candidates only.

In [ ]:
import platform
from collections import Counter
from datetime import UTC, datetime
from hashlib import sha256
from pathlib import Path
from urllib.request import HTTPRedirectHandler, build_opener

import astropy
import pyvo
from astropy.coordinates import SkyCoord
from astropy.io.votable import parse
from cosmic_astronomy.irsa import build_sia_url

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
raw = root / "data" / "raw" / "wise-0855-sia.vot"
target = SkyCoord("08h55m10.831s", "-7d14m42.53s", frame="icrs")
assert abs(target.ra.deg - 133.795129) < 0.0000005
assert abs(target.dec.deg - (-7.245147)) < 0.0000005
collection = "spherex_dr1"
url = build_sia_url(round(target.ra.deg, 6), round(target.dec.deg, 6), 0.02, collection)
print(platform.python_version(), astropy.__version__, pyvo.__version__)
print(target.to_string("decimal", precision=9), url)

In [ ]:
class RefuseRedirects(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


refresh = False
if refresh or not raw.exists():
    opener = build_opener(RefuseRedirects())
    limit = 8 * 1024 * 1024
    with opener.open(url, timeout=30) as response:
        payload = response.read(limit + 1)
    if len(payload) > limit:
        raise ValueError("IRSA response exceeded the inventory size limit")
    temporary = raw.with_suffix(".part")
    temporary.write_bytes(payload)
    try:
        candidate = parse(temporary)
        statuses = [info.value for info in candidate.iter_info() if info.name == "QUERY_STATUS"]
        if statuses != ["OK"]:
            raise ValueError(f"Unexpected IRSA QUERY_STATUS: {statuses}")
        temporary.replace(raw)
    finally:
        temporary.unlink(missing_ok=True)

votable = parse(raw)
statuses = [info.value for info in votable.iter_info() if info.name == "QUERY_STATUS"]
if statuses != ["OK"]:
    raise ValueError(f"Unexpected IRSA QUERY_STATUS: {statuses}")
table = votable.get_first_table().to_table(use_names_over_ids=True)
print("File modification time (UTC):", datetime.fromtimestamp(raw.stat().st_mtime, UTC))
print("SHA-256:", sha256(raw.read_bytes()).hexdigest())
print("Columns:", table.colnames)
print("Rows / distinct start MJDs:", len(table), len(set(table["t_min"])))
unique = {str(row["obs_publisher_did"]): row for row in table}
print(
    "Unique products by detector:",
    Counter(str(row["energy_bandpassname"]) for row in unique.values()),
)
print(table["obs_id", "obs_publisher_did", "energy_bandpassname", "t_min", "access_url"][:3])

In [ ]:
import json
from urllib.parse import urlencode, urlsplit

import numpy as np
from astropy.io import fits
from astropy.time import Time
from astropy.wcs import WCS
from PIL import Image
from reproject import reproject_interp
from scipy.interpolate import RegularGridInterpolator

# D2 ties for the largest inventory. These visits have nearly identical pointing
# and the measured local passbands agree. The initial outer epochs did not.
selected_ids = ("2025W19_1B_0181_1", "2026W20_1A_0728_1")
products = {
    str(row["obs_id"]): row for row in unique.values() if row["energy_bandpassname"] == "SPHEREx-D2"
}
selected = tuple(products[obs_id] for obs_id in selected_ids)
assert selected[1]["t_min"] - selected[0]["t_min"] > 1
size_deg = 0.05
side = 30
# Two bright field sources, picked by inspection of the first visit.
references = SkyCoord([133.79735, 133.81254], [-7.25849, -7.26167], unit="deg")
centroids = []
psf_fwhm = []
grid = WCS(naxis=2)
grid.wcs.ctype = ["RA---TAN", "DEC--TAN"]
grid.wcs.crval = [target.ra.deg, target.dec.deg]
grid.wcs.crpix = [(side + 1) / 2] * 2
grid.wcs.cdelt = [-6 / 3600, 6 / 3600]
grid.wcs.radesys = "ICRS"

frames = []
samples = []
images = []
for index, row in enumerate(selected, 1):
    access = str(row["access_url"])
    uri = urlsplit(access)
    if uri.scheme != "https" or uri.netloc != "irsa.ipac.caltech.edu" or uri.query or uri.fragment:
        raise ValueError(f"Unexpected IRSA cutout endpoint: {uri.netloc}")
    if not uri.path.startswith("/ibe/data/spherex/dr1/level2/") or not uri.path.endswith(".fits"):
        raise ValueError("Expected a DR1 spectral image FITS URL")
    cutout_url = (
        access
        + "?"
        + urlencode(
            {
                "center": f"{target.ra.deg:.6f},{target.dec.deg:.6f}",
                "size": size_deg,
            },
            safe=",",
        )
    )
    path = raw.parent / ("180as-" + Path(uri.path).name)
    if not path.exists():
        limit = 16_000_000
        with build_opener(RefuseRedirects()).open(cutout_url, timeout=45) as response:
            payload = response.read(limit + 1)
        if len(payload) > limit:
            raise ValueError("IRSA cutout exceeded the size limit")
        temporary = path.with_suffix(".part")
        temporary.write_bytes(payload)
        try:
            with fits.open(temporary, memmap=False) as hdus:
                hdus.verify("exception")
                if hdus["IMAGE"].data.ndim != 2:
                    raise ValueError("Expected a 2D scientific image")
            temporary.replace(path)
        finally:
            temporary.unlink(missing_ok=True)

    with fits.open(path, memmap=False) as hdus:
        hdus.verify("exception")
        image = hdus["IMAGE"]
        header = image.header
        flags = hdus["FLAGS"].data
        wcs = WCS(header, hdus).celestial
        x, y = wcs.world_to_pixel(target)
        height, width = image.data.shape
        center = wcs.pixel_to_world((width - 1) / 2, (height - 1) / 2)
        assert center.separation(target).deg < 0.02
        assert 0 <= x < width and 0 <= y < height
        assert wcs.pixel_to_world(x, y).separation(target).arcsec < 0.001
        assert header["OBSID"] == row["obs_id"]
        assert f"SPHEREx-D{header['DETECTOR']}" == row["energy_bandpassname"]
        assert abs(header["MJD-BEG"] - row["t_min"]) <= 0.0000005
        assert header["TIMESYS"] == "UTC"
        # DR1 alternate spectral WCS uses identity increments and a two-value lookup.
        assert header["CDELT1W"] == header["CDELT2W"] == 1
        assert header["CUNIT1W"] == header["CUNIT2W"] == "um"
        wave_table = hdus["WCS-WAVE"].data[0]
        lookup = RegularGridInterpolator((wave_table["Y"], wave_table["X"]), wave_table["VALUES"])
        wavelength, bandwidth = lookup(
            (
                y + 1 - header["CRPIX2W"] + header["CRVAL2W"],
                x + 1 - header["CRPIX1W"] + header["CRVAL1W"],
            )
        )
        samples.append((float(wavelength), float(bandwidth)))
        assert header["BUNIT"] == "MJy / sr"
        psf_fwhm.append(header["PSF_FWHM"])
        # IPAC defines bit 21 as SOURCE. Keep source-marked pixels when all other bits are clear.
        valid = ((flags & ~(1 << 21)) == 0) & np.isfinite(image.data)
        iy, ix = round(float(y)), round(float(x))
        assert valid[iy, ix], "The target pixel is masked"
        data = np.where(valid, image.data, np.nan)
        positions = []
        for reference in references:
            rx, ry = (round(float(v)) for v in wcs.world_to_pixel(reference))
            cut = data[ry - 1 : ry + 2, rx - 1 : rx + 2]
            assert cut.shape == (3, 3) and np.isfinite(cut).all()
            weights = np.clip(cut - np.median(data[valid]), 0, None)
            assert weights[1, 1] == weights.max() and weights.sum() > 0
            yy, xx = np.indices(cut.shape) - 1
            positions.append(
                wcs.pixel_to_world(
                    rx + np.sum(xx * weights) / weights.sum(),
                    ry + np.sum(yy * weights) / weights.sum(),
                )
            )
        centroids.append(positions)
        aligned, _ = reproject_interp(
            (data, wcs), grid, shape_out=(side, side), order="nearest-neighbor"
        )
        images.append(aligned)
        name = f"wise-0855-{index}"
        frames.append(
            {
                "id": name,
                "obs_id": header["OBSID"],
                "mjd": header["MJD-BEG"],
                "time_scale": "UTC",
                "iso_time": Time(header["MJD-BEG"], format="mjd", scale="utc").isot + "Z",
                "detector": str(row["energy_bandpassname"]),
                "band": None,
                "hdu": hdus.index_of("IMAGE"),
                "flags": ["quality-flags-masked-except-source"],
                "preview_file": name + ".png",
                "sha256": sha256(path.read_bytes()).hexdigest(),
            }
        )
        print(
            cutout_url,
            [hdu.name for hdu in hdus],
            image.shape,
            header["BUNIT"],
            "target wavelength/bandwidth (um):",
            samples[-1],
            "unflagged pixels:",
            int(valid.sum()),
        )

# Display criteria for this pilot: local centroid and bandwidth agreement within
# 10% of the narrower bandwidth, median PSF FWHM within 10%, and both reference
# centroid shifts within half of the 6-arcsecond display pixel.
spectral_tolerance = 0.1 * min(samples[0][1], samples[1][1])
assert np.max(np.abs(np.subtract(*samples))) < spectral_tolerance
assert abs(psf_fwhm[0] - psf_fwhm[1]) < 0.1 * min(psf_fwhm)
residuals = [first.separation(second).arcsec for first, second in zip(*centroids, strict=True)]
assert max(residuals) < 3
coverage = np.logical_and(*[np.isfinite(image) for image in images]).mean()
assert coverage >= 0.7
print("Reference shifts (arcsec):", residuals, "shared coverage:", coverage)
low, high = np.nanpercentile(np.stack(images), (5, 95))
if not np.isfinite([low, high]).all() or high <= low:
    raise ValueError("No usable shared preview stretch")
output = root / "apps" / "web" / "public" / "previews"
output.mkdir(parents=True, exist_ok=True)
warnings = [
    "Previews use a shared linear stretch from "
    f"{low:.6g} to {high:.6g} MJy/sr. Quality flags are masked except SOURCE.",
]
for frame, image in zip(frames, images, strict=True):
    scaled = np.clip((image - low) / (high - low), 0, 1)
    luminance = np.nan_to_num(scaled, nan=0.0)
    rgba = np.stack((luminance, luminance, luminance, np.isfinite(scaled)), axis=-1)
    pixels = np.round(255 * rgba[::-1]).astype(np.uint8)
    preview = output / frame["preview_file"]
    Image.fromarray(pixels).resize((512, 512), Image.Resampling.NEAREST).save(preview)
    frame["flags"].append("preview-sha256:" + sha256(preview.read_bytes()).hexdigest())

manifest = {
    "slug": "wise-0855",
    "name": "WISE 0855",
    "ra_deg": round(target.ra.deg, 6),
    "dec_deg": round(target.dec.deg, 6),
    "coord_source": "NASA Space Apps challenge statement via .documentation/07-MATEO-DATOS.md",
    "coord_epoch": None,
    "collection": collection,
    "radius_deg": 0.02,
    "queried_at": datetime.fromtimestamp(raw.stat().st_mtime, UTC).isoformat(),
    "comparable": True,
    "comparison_note": (
        f"The two DR1 D2 exposures sample approximately {samples[0][0]:.3f} micrometers "
        f"at the specified position, and two field reference centroids agree to "
        f"within {max(residuals):.2f} arcseconds on the shared WCS grid. "
        "The previews use a common linear stretch for visual comparison. "
        "Target motion and flux changes were not measured."
    ),
    "warnings": warnings,
    "frames": frames,
}

destination = root / "data" / "manifests" / "wise-0855.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(manifest, indent=2) + chr(10), encoding="utf-8")
print(destination)